# VOICE-CUE 병합본 업로드 (허깅페이스)

이미 끝난 학습 실행(본인 것이든, 남의 공개 노트북이든)의 병합본을 허깅페이스로 올린다.
GPU가 필요 없어서 학습 노트북(`kaggle_train.ipynb`)과 분리했다 — Accelerator는 None으로
둬도 된다.

**시작 전에**:
1. 우측 **Add Input** → 병합본이 들어있는 노트북을 검색해서 붙인다. 본인 것이든 남의
   공개(Public) 노트북이든 방식은 같다. 비공개인 남의 노트북은 그 사람이 협업자로
   초대해야만 보인다.
2. 붙인 뒤 왼쪽 **Input** 패널에서 `finetune/out/merged`까지 들어가 파일이 보이는지
   확인하고, 정확한 경로를 복사해서 아래 1번 셀의 `MERGED_DIR`에 붙여넣는다.
3. 우측 **Settings → Internet → On** (업로드에 필요).
4. **Add-ons → Secrets** 에서 `HF_TOKEN`이라는 이름으로 대상 저장소에 Write 권한이 있는
   허깅페이스 토큰을 등록하고, 이 노트북에 **Attach**.

학습이 아니라 파일 복사·업로드라 몇 분이면 끝난다 — Save & Run All 안 쓰고 셀을
위에서부터 순서대로 그냥 실행(Shift+Enter)하면 된다.

**이 노트북이 반영하는 것**: 배포 중 겪었던 문제들 중 지금 시점에 고칠 수 있는 건
"transformers가 채팅 템플릿을 tokenizer_config.json이 아니라 별도 chat_template.jinja로
저장해서, TGI/vLLM/Ollama가 `Template error: template not found`로 실패하는 것" 하나뿐이라
그것만 자동으로 고친다(3번 섹션). 나머지(엔드포인트를 Default 컨테이너로 만듦 / 403 권한
부족 / 503 콜드스타트 / response_format 거절)는 전부 엔드포인트 생성·설정 쪽 문제라 업로드
스크립트로는 못 고치며, 앱 코드(`modules/llm_engine.py`)가 이미 대응하고 있다.


## 1. 설정

In [ ]:
from pathlib import Path

# 왼쪽 Input 패널에서 확인한 정확한 경로로 바꾸세요.
MERGED_DIR = Path("/kaggle/input/노트북-이름/finetune/out/merged")

# 올릴 허깅페이스 저장소 (본인 계정이어야 한다 -- HF_TOKEN이 본인 계정 권한이므로).
REPO_ID = "계정이름/voicecue-qwen2.5-3b"
PRIVATE = False


## 2. 병합본 확인

In [ ]:
assert MERGED_DIR.is_dir(), (
    f"{MERGED_DIR} 가 없습니다 -- 왼쪽 Input 패널에서 실제 경로를 다시 확인하세요."
)
weight_files = list(MERGED_DIR.glob("*.safetensors")) + list(MERGED_DIR.glob("*.bin"))
assert weight_files, f"{MERGED_DIR} 안에 가중치 파일이 없습니다. 병합이 실패했을 수 있습니다."

print(f"가중치 파일 {len(weight_files)}개 확인됨:")
for f in sorted(weight_files):
    print(f"  {f.name}  ({f.stat().st_size / 1e9:.2f} GB)")


## 3. 채팅 템플릿 점검·보정

transformers 4.56부터 `save_pretrained`가 채팅 템플릿을 `tokenizer_config.json` 안이
아니라 별도 `chat_template.jinja` 파일로 저장한다. 그런데 TGI·vLLM·Ollama는 전부
`tokenizer_config.json`의 `chat_template` 키만 읽으므로, 그 상태로 올리면 나중에
`/v1/chat/completions` 요청이 전부 `Template error: template not found`(422)로 실패한다
— 실제로 이 프로젝트에서 겪은 문제다.

이 병합본을 만든 `train_lora.py`가 최신 버전이면 이미 심어져 있을 수 있는데, 그래도 한
번 더 확인하고 없으면 여기서 고친다. Input 폴더는 읽기 전용이라 파일을 직접 고칠 수는
없어서, 고친 내용은 메모리에만 들고 있다가 5번 셀에서 그 파일 하나만 따로 올린다.


In [ ]:
import json

cfg_path = MERGED_DIR / "tokenizer_config.json"
jinja_path = MERGED_DIR / "chat_template.jinja"
patched_tokenizer_config = None  # None이면 5번 셀에서 원본 그대로 올라간다.

if not cfg_path.exists():
    print("경고: tokenizer_config.json이 없습니다. 이 폴더가 진짜 병합본이 맞는지 확인하세요.")
else:
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    if cfg.get("chat_template"):
        print("이미 tokenizer_config.json에 chat_template이 들어 있습니다. 손댈 것 없음.")
    elif jinja_path.exists():
        cfg["chat_template"] = jinja_path.read_text(encoding="utf-8")
        patched_tokenizer_config = json.dumps(cfg, ensure_ascii=False, indent=2).encode("utf-8")
        print("chat_template.jinja 내용을 심었습니다 -- 5번 셀 업로드 때 반영됩니다.")
    else:
        print(
            "경고: chat_template.jinja도 없어 템플릿을 못 심었습니다. "
            "TGI 등에서 채팅 요청이 실패할 수 있습니다."
        )


## 4. 준비 확인

In [ ]:
!pip install -q -U huggingface_hub


## 5. 업로드

In [ ]:
import io

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)
api.create_repo(REPO_ID, exist_ok=True, private=PRIVATE)

# tokenizer_config.json을 고쳤으면 그 파일만 빼고 나머지를 통째로 올린 뒤, 고친 버전을
# 따로 올린다. Input 폴더는 읽기 전용이라 폴더 자체를 고쳐서 올릴 수는 없다.
ignore = ["tokenizer_config.json"] if patched_tokenizer_config else None
api.upload_folder(
    folder_path=str(MERGED_DIR),
    repo_id=REPO_ID,
    commit_message="Kaggle 병합본 업로드",
    ignore_patterns=ignore,
)

if patched_tokenizer_config:
    api.upload_file(
        path_or_fileobj=io.BytesIO(patched_tokenizer_config),
        path_in_repo="tokenizer_config.json",
        repo_id=REPO_ID,
        commit_message="Embed chat template for inference servers",
    )

print(f"업로드 완료: https://huggingface.co/{REPO_ID}")
print(
    "HF Inference Endpoint를 쓰고 있다면 Settings > Advanced > commit revision을 "
    "방금 올린 커밋으로 바꿔야 반영됩니다 (Pause/Resume만으로는 이전 캐시를 그대로 씁니다)."
)
